In [1]:
%pip install -q --upgrade pip

# Install a ragas version that supports langchain-core >=0.3,.
# plus a compatible langchain-community pin to avoid the ChatVertexAI import error.
%pip install -q \
    "ragas>=0.2.15" \
    "langchain-google-genai>=2.0.0" \
    "langchain-community<0.4.2" \
    langchain_cohere \
    langchain_core \
    datasets pandas matplotlib seaborn

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from ragas import evaluate, EvaluationDataset 
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
from ragas.dataset_schema import SingleTurnSample
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_cohere import ChatCohere
from ragas.llms import LangchainLLMWrapper
from getpass import getpass

/tmp/ipykernel_1045/1040900535.py:7: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_1045/1040900535.py:7: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_1045/1040900535.py:7: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summarizat

In [5]:
COHERE_API_KEY = getpass("Enter your Cohere API key: ")

In [4]:
GOOGLE_API_KEY = getpass("Enter your GOOGLE API key: ")

In [17]:
cohere_llm = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model="command-a-03-2025",
    temperature=0,
    max_tokens=4096,          # RAGAS rubrics can be verbose
)

# Wrap for RAGAS (required for evaluate() / metric-level llm param
# judge_llm = LangchainLLMWrapper(cohere_llm, is_finished_parser=cohere_is_finished_parser,)

In [32]:
import cohere

cohere_client = cohere.ClientV2(COHERE_API_KEY)

In [33]:
from ragas.llms import llm_factory

judge_llm = llm_factory(
    "command-a-03-2025",           # model name
    provider="cohere",             # tell RAGAS this is Cohere
    client=cohere_client,          # pass the raw Cohere client
    adapter="litellm",             # <-- the key: force LiteLLM adapter
    temperature=0,
    max_tokens=8192,
)

In [ ]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash-lite",
#     response_mime_type="application/json",  # forces raw JSON, no fences
#     google_api_key="GOOGLE_API_KEY",
# )

embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",  # or "models/embedding-001"
        google_api_key=GOOGLE_API_KEY,
    )
)

/tmp/ipykernel_1045/2791003649.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(


In [12]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train").select(range(1))
df = ds.to_pandas()

# Show the columns and a sample row
print(df.columns.tolist())
df.head(2)

['user_input', 'reference', 'base_response', 'ft_response']


,user_input,reference,base_response,ft_response
0,Original Post: Help with Small living room Use...,The OP asked about general design suggestions ...,"The original poster, moving into a 1920s craft...",The user is asking for advice on how to arrang...


In [14]:
def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [ ]:
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision

# Instantiate each metric with the judge LLM
faithfulness_metric = Faithfulness(llm=judge_llm)
answer_relevancy_metric = AnswerRelevancy(llm=judge_llm, embeddings=embeddings)
context_precision_metric = ContextPrecision(llm=judge_llm)

# TODO: modern embeddings

ValueError: Collections metrics only support modern embeddings. Found: LangchainEmbeddingsWrapper. Use: embedding_factory('openai', model='text-embedding-ada-002', client=openai_client, interface='modern')

In [ ]:
# Define the metrics we want to compute
# metrics = [
#     SummarizationScore(llm=judge_llm, coeff=0.5),  # 0.5 balances QA vs. conciseness
#     SemanticSimilarity(embeddings=embeddings),
#     AnswerCorrectness(llm=judge_llm, embeddings=embeddings),
# ]

base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: GoogleGenerativeAIError(Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}})
ERROR:ragas.executor:Exception raised in Job[2]: OutputParserException(Failed to parse StringIO from completion {"TP": [{"statement": "The original poster is moving into a 1920s craftsman home.", "reason": "This is directly supported by the ground truth, which mentions the original poster's new living room in a 1920s craftsman home."}, {"statement": "The original poster sought advice on laying out their small 

In [ ]:
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
)

# Convert results to dataframes for easy inspection
base_df = base_result.to_pandas()
ft_df   = ft_result.to_pandas()

print("Base model scores:")
print(base_df.mean(numeric_only=True))
print("\nFine‑tuned model scores:")
print(ft_df.mean(numeric_only=True))